# AIOps: Predictive Analysis of CI/CD Pipeline Failures
## Pipeline Structure

```
Phase 0  ── Install & imports
Phase 1  ── Raw load (ALL columns, chunked, float32 downcast)
Phase 2  ── RAW DATA INSIGHTS
            2A  Shape / dtypes / column inventory
            2B  df.info() style output
            2C  Random sample rows
            2D  Numeric describe (skew, kurtosis, CV)
            2E  Categorical value counts & cardinality
            2F  Missing values (count, %, heatmap, bar)
            2G  Duplicate detection (subset-key aware)
            2H  Zero / near-zero variance
            2I  Target distribution
            2J  Temporal coverage — builds per month
            2K  Repository build-count distribution
Phase 3  ── VISUALISATION ANALYSIS (50k sample, pre-preprocessing)
            3A  Class distribution bar + pie
            3B  Violin plots — raw features vs target
            3C  KDE density plots
            3D  Count plots — categorical features
            3E  Off-hour / weekend failure bars
            3F  Outlier boxplots (zero-inflation aware IQR)
            3G  Pair plot (10k sample)
            3H  Temporal failure rate by hour & day
Phase 4  ── Preprocessing
Phase 5  ── Post-preprocessing correlation
            5A  Pearson correlation heatmap (cleaned df)
            5B  Feature → target correlation bar
            5C  Temporal distribution shift report
Phase 6  ── XGBoost Feature Engineering (22 features)
Phase 7  ── Post-Feature-Engineering Insights
            7A  Engineered feature describe
            7B  Engineered feature correlation heatmap
            7C  Violin plots vs target
            7D  Feature → target ranking
Phase 8  ── Chronological split 70 / 15 / 15
Phase 9  ── XGBoost Training
            9A  Train with EarlyStopping
            9B  Loss & AUC learning curves per round
            9C  Evaluation + optimal threshold selection
            9D  ROC + Precision-Recall curves
            9E  Threshold sweep analysis
            9F  Feature importance — Gain / Cover / Weight
            9G  Save model
Phase 10 ── LSTM Sequence Engineering
Phase 11 ── BiLSTM Training
            11A  Build & train BiLSTM
            11B  4-panel learning curves (Loss/AUC/Precision/Recall)
            11C  Evaluation
            11D  ROC + Precision-Recall curves
            11E  Confusion matrix
            11F  Save model + scaler
Phase 12 ── Fair Side-by-Side Comparison
```


## Phase 0 — Install & Imports

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
pip("xgboost>=2.0", "scikit-learn", "pandas", "numpy",
    "matplotlib", "seaborn", "tensorflow>=2.13", "psutil", "joblib")
print("All packages ready.")


In [ ]:
import gc, os, warnings, joblib
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import tensorflow as tf

from sklearn.metrics       import (classification_report, roc_auc_score,
                                    matthews_corrcoef, ConfusionMatrixDisplay,
                                    precision_recall_curve, roc_curve,
                                    average_precision_score, log_loss,
                                    f1_score, precision_score, recall_score,
                                    accuracy_score)
from sklearn.preprocessing import StandardScaler
from xgboost               import XGBClassifier

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

RAM_BUDGET_GB = 20.0
DATASET       = "final-2017-01-25.csv"
TIMESTAMP     = "gh_build_started_at"
TARGET        = "tr_status"
SAMPLE_N      = 50_000
PAIR_N        = 10_000

# ── Unique key columns — used for smart deduplication ─────────────
DEDUP_SUBSET  = ["gh_project_name", "gh_build_started_at", "git_trigger_commit"]

def ram_gb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

def mem_check(label: str = "") -> None:
    used = ram_gb()
    bar  = (" " * int(used / RAM_BUDGET_GB * 30)).ljust(30, " ")
    flag = " " if used < RAM_BUDGET_GB else "OVER BUDGET"
    print(f"[RAM] {label:<48s} {used:5.2f}/{RAM_BUDGET_GB:.0f} GB  {flag}  |{bar}|")
    assert used < RAM_BUDGET_GB, f"RAM exceeded: {used:.2f} GB"

mem_check("startup")


## Phase 1 — Raw Data Load (All Columns)

Chunked at 150 000 rows. `float64→float32`, `int64→smallest int` downcast
applied per chunk to stay within RAM budget.


In [ ]:
CHUNK = 150_000

def downcast_chunk(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes("int64").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

chunks = []
for i, chunk in enumerate(
        pd.read_csv(DATASET, parse_dates=[TIMESTAMP],
                    chunksize=CHUNK, low_memory=True)):
    chunks.append(downcast_chunk(chunk))
    if (i + 1) % 5 == 0:
        mem_check(f"raw load — chunk {i+1}")

raw_df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

mem_check("raw load complete")
print(f"\nRaw shape : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print(f"Memory    : {raw_df.memory_usage(deep=True).sum()/1e9:.2f} GB")


---
## Phase 2 — Raw Data Insights

### 2A — Shape, Dtypes & Column Inventory

In [ ]:
print("=" * 65)
print(f"  SHAPE : {raw_df.shape[0]:,} rows  ×  {raw_df.shape[1]} columns")
print("=" * 65)
print("\nDtype breakdown:")
print(raw_df.dtypes.value_counts().to_string())
print(f"\nTotal memory : {raw_df.memory_usage(deep=True).sum()/1e9:.3f} GB")

col_info = pd.DataFrame({
    "dtype"     : raw_df.dtypes,
    "non_null"  : raw_df.notna().sum(),
    "null_cnt"  : raw_df.isna().sum(),
    "null_pct"  : (raw_df.isna().mean()*100).round(2),
    "n_unique"  : raw_df.nunique(),
    "sample_val": [str(raw_df[c].dropna().iloc[0])[:35]
                   if raw_df[c].notna().any() else "ALL NULL"
                   for c in raw_df.columns],
})
print(f"\n── Full column inventory ({len(col_info)} columns) ──")
print(col_info.to_string())


### 2B — df.info() Style Output

In [ ]:
print(f"<class 'pandas.core.frame.DataFrame'>")
print(f"RangeIndex: {len(raw_df):,} entries")
print(f"Columns: {raw_df.shape[1]} total")
print(f"{'#':<5} {'Column':<42} {'Non-Null':>14}  {'Dtype'}")
print("-"*72)
for i, col in enumerate(raw_df.columns):
    nn = raw_df[col].notna().sum()
    print(f"{i:<5} {col:<42} {nn:>12,}   {raw_df[col].dtype}")
print(f"\nMemory usage: {raw_df.memory_usage(deep=True).sum()/1e6:.1f}+ MB")


### 2C — Random Sample Rows

In [ ]:
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 28)

print("── 5 random rows ──")
display(raw_df.sample(5, random_state=42))

print("\n── Head 3 ──")
display(raw_df.head(3))

print("\n── Tail 3 ──")
display(raw_df.tail(3))

KEY_COLS = [TARGET, TIMESTAMP, "gh_project_name", "git_branch",
            "git_diff_src_churn", "gh_sloc", "gh_team_size",
            "gh_num_pr_comments", "gh_is_pr", "git_trigger_commit"]
avail = [c for c in KEY_COLS if c in raw_df.columns]
print("\n── Random 8 rows — key columns ──")
display(raw_df[avail].sample(8, random_state=7))


### 2D — Numeric Descriptive Statistics

In [ ]:
num_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {len(num_cols)}\n")

desc = raw_df[num_cols].describe(percentiles=[.01,.05,.25,.50,.75,.95,.99]).T
desc["skewness"] = raw_df[num_cols].skew().round(3)
desc["kurtosis"] = raw_df[num_cols].kurtosis().round(3)
desc["cv"]       = (desc["std"] / (desc["mean"].abs() + 1e-9)).round(3)
desc["zero_pct"] = ((raw_df[num_cols] == 0).mean()*100).round(2)

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 280)
print("── Full describe + skew + kurtosis + zero% ──")
print(desc.to_string())

print("\n── High-skewness (|skew|>2) — log-transform candidates ──")
hs = desc[desc["skewness"].abs()>2].sort_values("skewness", ascending=False)
print(hs[["mean","std","min","50%","max","skewness","zero_pct"]].to_string())


### 2E — Categorical Columns: Cardinality & Value Counts

In [ ]:
cat_cols = raw_df.select_dtypes(include=["object","bool","category"]).columns.tolist()
print(f"Categorical / bool columns: {len(cat_cols)}\n")

for col in cat_cols:
    vc  = raw_df[col].value_counts(dropna=False)
    pct = raw_df[col].value_counts(normalize=True, dropna=False)*100
    print(f"\n── {col}  (unique={raw_df[col].nunique()}, "
          f"null={raw_df[col].isna().sum():,}) ──")
    tbl = pd.DataFrame({"count":vc,"pct%":pct.round(2)}).head(12)
    print(tbl.to_string())

print(f"\n══ TARGET: {TARGET} — full distribution ══")
vc_t = raw_df[TARGET].value_counts()
for v, n in vc_t.items():
    bar = "█" * int(n/len(raw_df)*60)
    print(f"  {v:<12}  {n:>9,}  ({n/len(raw_df)*100:5.2f}%)  {bar}")


### 2F — Missing Value Analysis

In [ ]:
null_df = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_pct"  : (raw_df.isna().mean()*100).round(3),
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

print(f"Columns with ANY missing : {len(null_df)} / {raw_df.shape[1]}")
print(f"Fully complete columns   : {raw_df.shape[1]-len(null_df)}")
print()
print(null_df.to_string())

# Heatmap
cols_miss = null_df.index.tolist()
if cols_miss:
    _s = raw_df[cols_miss].sample(min(10_000,len(raw_df)), random_state=42)
    fig, axes = plt.subplots(1, 2, figsize=(max(14,len(cols_miss)*0.45)+6, 4))
    fig.suptitle("Missing Value Analysis", fontweight="bold", fontsize=13)

    sns.heatmap(_s.isna(), cbar=False, yticklabels=False,
                cmap="YlOrRd", ax=axes[0])
    axes[0].set_title("Heatmap — Yellow = Missing", fontweight="bold")
    axes[0].set_xticklabels(axes[0].get_xticklabels(),
                             rotation=45, ha="right", fontsize=7)

    null_df["missing_pct"].plot.barh(ax=axes[1], color="#EF5350", edgecolor="white")
    axes[1].set_title("Missing % per Column", fontweight="bold")
    axes[1].set_xlabel("Missing %")
    axes[1].axvline(50, color="black", ls="--", lw=1, label="50%")
    axes[1].legend(); axes[1].grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig("da_missing.png", dpi=130, bbox_inches="tight")
    plt.show()
    del _s; gc.collect()

    # Flag 100% null columns — these will be dropped
    full_null = null_df[null_df["missing_pct"] >= 99.0]
    print(f"\n  Columns with ≥99% missing (will be auto-dropped): {len(full_null)}")
    print(full_null.to_string())
else:
    print("\n  No missing values.")


### 2G — Duplicate Detection (Subset-Key Aware)

>  `drop_duplicates()` on all 66 columns removed 2.95M rows (76%)
> because unique ID columns (`tr_build_id`) made every row look unique — yet
> the operation still ran and silently corrupted the dataset.
>
>  Check duplicates on the meaningful business key:
> `(gh_project_name, gh_build_started_at, git_trigger_commit)`.
> Full-row exact duplicates are still reported separately.


In [ ]:
# ── 2G. Smart duplicate detection ────────────────────────────────────

# Full-row duplicates (exact match on ALL columns)
n_full_dup = raw_df.duplicated().sum()
print(f"Full-row exact duplicates   : {n_full_dup:,}  "
      f"({n_full_dup/len(raw_df)*100:.4f}%)")

# Business-key duplicates (meaningful dedup)
dedup_cols_avail = [c for c in DEDUP_SUBSET if c in raw_df.columns]
if dedup_cols_avail:
    n_key_dup = raw_df.duplicated(subset=dedup_cols_avail).sum()
    print(f"\nBusiness-key duplicates     : {n_key_dup:,}  "
          f"({n_key_dup/len(raw_df)*100:.4f}%)")
    print(f"  Key used : {dedup_cols_avail}")
    if n_key_dup > 0:
        print("\nSample business-key duplicate rows:")
        display(raw_df[raw_df.duplicated(subset=dedup_cols_avail, keep=False)].head(4))
    else:
        print("  No business-key duplicates found.")
else:
    print("\nDedup key columns not all present; skipping subset dedup.")

print(f"\nTotal rows : {len(raw_df):,}")
print(f"Will keep  : {len(raw_df)-n_full_dup:,} after full-row dedup "
      f"(only {n_full_dup:,} to remove)")


### 2H — Zero-Variance & Near-Zero-Variance Features

In [ ]:
_num = raw_df.select_dtypes(include=[np.number]).columns
var_df = pd.DataFrame({
    "variance"  : raw_df[_num].var(),
    "std"       : raw_df[_num].std(),
    "n_unique"  : raw_df[_num].nunique(),
    "unique_pct": (raw_df[_num].nunique()/len(raw_df)*100).round(5),
    "zero_pct"  : ((raw_df[_num]==0).mean()*100).round(2),
}).sort_values("variance")

zero_var  = var_df[var_df["variance"]==0]
near_zero = var_df[(var_df["variance"]>0) & (var_df["unique_pct"]<0.01)]

print(f"Zero-variance columns      : {len(zero_var)}")
print(zero_var.to_string() if len(zero_var) else "  (none)")
print(f"\nNear-zero-variance columns : {len(near_zero)}")
print(near_zero.to_string() if len(near_zero) else "  (none)")

# Store zero-variance col names for later use in feature engineering guard
ZERO_VAR_COLS = zero_var.index.tolist()
print(f"\nZero-variance columns to guard in feature eng: {ZERO_VAR_COLS}")

fig, ax = plt.subplots(figsize=(10,5))
var_df.head(20)["variance"].plot.barh(ax=ax, color="#AB47BC", edgecolor="white")
ax.set_title("Bottom 20 Features by Variance", fontweight="bold")
ax.set_xlabel("Variance"); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("da_low_variance.png", dpi=130, bbox_inches="tight")
plt.show()


### 2I — Target Variable Distribution

In [ ]:
vc    = raw_df[TARGET].value_counts()
nc    = raw_df[raw_df[TARGET]!="canceled"]
fail  = nc[TARGET].isin(["failed","errored"]).sum()
total = len(nc)

print("Raw tr_status distribution:")
for v in vc.index:
    bar = "█"*int(vc[v]/len(raw_df)*60)
    print(f"  {v:<12}  {vc[v]:>9,}  ({vc[v]/len(raw_df)*100:5.2f}%)  {bar}")

print(f"\nAfter removing 'canceled' ({(raw_df[TARGET]=='canceled').sum():,} rows):")
print(f"  Passed         : {vc.get('passed',0):>9,}  ({vc.get('passed',0)/total*100:.2f}%)")
print(f"  Failed+Errored : {fail:>9,}  ({fail/total*100:.2f}%)")
print(f"  Imbalance ratio: {vc.get('passed',0)/fail:.3f} : 1")
print(f"  Optimal scale_pos_weight for XGBoost: {vc.get('passed',0)/fail:.4f}")

fig, axes = plt.subplots(1,2,figsize=(12,4))
fig.suptitle("Target Variable Distribution", fontweight="bold", fontsize=13)
colors = ["#43A047","#E53935","#FB8C00","#9E9E9E","#5C6BC0"]
axes[0].bar(vc.index, vc.values, color=colors[:len(vc)], edgecolor="white")
axes[0].set_title("All raw statuses", fontweight="bold"); axes[0].set_ylabel("Count")
for bar, v in zip(axes[0].patches, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                 f"{v:,}", ha="center", fontsize=8)
vals = [total-fail, fail]
axes[1].pie(vals,
            labels=[f"Passed\n{vals[0]:,}", f"Failed/Errored\n{vals[1]:,}"],
            autopct="%1.2f%%", colors=["#43A047","#E53935"],
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Binary target (canceled removed)", fontweight="bold")
plt.tight_layout()
plt.savefig("da_target_distribution.png", dpi=130, bbox_inches="tight")
plt.show()
del nc; gc.collect()


### 2J — Temporal Coverage

In [ ]:
ts_s = raw_df[TIMESTAMP].dropna()
print(f"Date range : {ts_s.min()}  →  {ts_s.max()}")
print(f"Span       : {(ts_s.max()-ts_s.min()).days} days")

monthly = (raw_df.set_index(TIMESTAMP).resample("ME")[TARGET]
           .agg(total="count",
                failures=lambda x: x.isin(["failed","errored"]).sum()))
monthly["fail_rate"] = (monthly["failures"]/monthly["total"]*100).round(2)

fig, axes = plt.subplots(2,1,figsize=(14,7),sharex=True)
fig.suptitle("Monthly Build Activity & Failure Rate", fontweight="bold",fontsize=13)
axes[0].bar(monthly.index, monthly["total"],    color="#42A5F5",
            edgecolor="white", width=20, label="Total")
axes[0].bar(monthly.index, monthly["failures"], color="#EF5350",
            edgecolor="white", width=20, label="Failures")
axes[0].set_ylabel("Count"); axes[0].legend(); axes[0].grid(axis="y",alpha=0.3)
axes[1].plot(monthly.index, monthly["fail_rate"], color="#E53935", lw=2, marker="o", ms=4)
axes[1].axhline(monthly["fail_rate"].mean(), ls="--", color="gray", lw=1.5,
                label=f"Mean {monthly['fail_rate'].mean():.1f}%")
axes[1].set_ylabel("Failure Rate (%)"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("da_temporal_coverage.png", dpi=130, bbox_inches="tight")
plt.show()
print("\nMonthly stats:")
print(monthly.describe().round(2).to_string())


### 2K — Repository Build-Count Distribution

In [ ]:
repo_counts = raw_df["gh_project_name"].value_counts()
print(f"Total repositories: {len(repo_counts):,}")
print("\nBuilds per repo — summary:")
print(repo_counts.describe().round(1).to_string())

brackets = [(1,9),(10,49),(50,199),(200,999),(1000,9999),(10000,10**9)]
print("\n── Bracket analysis ──")
for lo,hi in brackets:
    n = ((repo_counts>=lo)&(repo_counts<=hi)).sum()
    tag = " ← LSTM eligible" if lo>=10 else " ← skipped (< 10 builds)"
    print(f"  {lo:>6}–{hi:<10} repos: {n:>5,}{tag}")

fig, axes = plt.subplots(1,2,figsize=(14,4))
fig.suptitle("Repository Build-Count Distribution", fontweight="bold",fontsize=13)
axes[0].hist(repo_counts.values, bins=80, color="#7E57C2", edgecolor="white")
axes[0].set_xlabel("Builds per repo"); axes[0].set_ylabel("Repo count")
axes[0].set_title("Linear scale"); axes[0].grid(axis="y",alpha=0.3)
axes[1].hist(repo_counts.values, bins=80, color="#7E57C2", edgecolor="white",log=True)
axes[1].set_xscale("log"); axes[1].set_xlabel("Builds per repo (log)")
axes[1].set_ylabel("Repo count (log)"); axes[1].set_title("Log-log scale")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("da_repo_distribution.png", dpi=130, bbox_inches="tight")
plt.show()
print("\nTop 10 repos by build count:")
print(repo_counts.head(10).to_string())


---
## Phase 3 — Visualisation Analysis (50k sample, pre-preprocessing)

All plots on a **50 000-row random sample** (canceled rows excluded).
A temporary `y_raw` binary label is created for plotting only.


In [ ]:
rng  = np.random.default_rng(42)
_nc  = raw_df[raw_df[TARGET]!="canceled"].copy()
idx  = rng.choice(len(_nc), size=min(SAMPLE_N,len(_nc)), replace=False)
sdf  = _nc.iloc[idx].copy()
sdf["y_raw"] = sdf[TARGET].isin(["failed","errored"]).astype("int8")
del _nc; gc.collect()
mem_check("50k sample built")
print(f"Sample: {sdf.shape}  |  failure rate: {sdf['y_raw'].mean():.2%}")


### 3A — Class Distribution

In [ ]:
counts = sdf["y_raw"].value_counts().sort_index()
fig, axes = plt.subplots(1,2,figsize=(11,4))
fig.suptitle("Class Distribution (50k sample)", fontweight="bold",fontsize=13)
axes[0].bar(["Passed (0)","Failed/Errored (1)"], counts.values,
            color=["#43A047","#E53935"], edgecolor="white", width=0.5)
axes[0].set_ylabel("Count"); axes[0].set_title("Bar Chart", fontweight="bold")
for bar,v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
                 f"{v:,}", ha="center", fontsize=10)
axes[1].pie(counts.values,
            labels=[f"Passed\n{counts[0]:,}",f"Failed/Errored\n{counts[1]:,}"],
            autopct="%1.2f%%", colors=["#43A047","#E53935"],
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Proportion", fontweight="bold")
plt.tight_layout()
plt.savefig("vis_class_dist.png", dpi=130, bbox_inches="tight"); plt.show()


### 3B — Violin Plots: Raw Features vs Target

In [ ]:
VIO_FEATS = ["git_diff_src_churn","gh_sloc","gh_team_size",
             "gh_num_pr_comments","gh_num_commits_in_push"]
vio_feats = [f for f in VIO_FEATS if f in sdf.columns
             and sdf[f].notna().sum() > 1000
             and f not in ZERO_VAR_COLS]

n = len(vio_feats)
fig, axes = plt.subplots(1, n, figsize=(4.5*n, 5))
if n == 1: axes = [axes]
fig.suptitle("Violin Plots — Raw Features vs Outcome (50k, log-scaled)",
             fontweight="bold", fontsize=13)
for ax, feat in zip(axes, vio_feats):
    _tmp = sdf[[feat,"y_raw"]].dropna().copy()
    _tmp[feat] = np.log10(_tmp[feat].clip(lower=0)+1)
    _tmp["Outcome"] = _tmp["y_raw"].map({0:"Passed",1:"Failed"})
    sns.violinplot(data=_tmp, x="Outcome", y=feat,
                   palette={"Passed":"#43A047","Failed":"#E53935"},
                   inner="box", cut=0, ax=ax, linewidth=0.8)
    ax.set_title(f"log10({feat}+1)", fontweight="bold", fontsize=9)
    ax.set_xlabel(""); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("vis_violin.png", dpi=130, bbox_inches="tight"); plt.show()


### 3C — KDE Density Plots

In [ ]:
KDE_FEATS = ["git_diff_src_churn","gh_sloc","gh_team_size","gh_num_pr_comments"]
kde_feats = [f for f in KDE_FEATS if f in sdf.columns
             and f not in ZERO_VAR_COLS
             and sdf[f].notna().sum() > 1000]

fig, axes = plt.subplots(1,len(kde_feats), figsize=(4.5*len(kde_feats),4))
if len(kde_feats)==1: axes=[axes]
fig.suptitle("KDE Density — Key Features by Outcome (50k, log-scaled)",
             fontweight="bold", fontsize=13)
for ax, feat in zip(axes, kde_feats):
    for label, col in [(0,"#43A047"),(1,"#E53935")]:
        _x = np.log10(sdf[sdf["y_raw"]==label][feat].dropna().clip(lower=0)+1)
        sns.kdeplot(_x, ax=ax, color=col, fill=True, alpha=0.35,
                    label="Passed" if label==0 else "Failed")
    ax.set_title(f"log10({feat}+1)", fontweight="bold",fontsize=9)
    ax.set_xlabel("log10 value"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("vis_kde.png", dpi=130, bbox_inches="tight"); plt.show()


### 3D — Categorical Count Plots & Off-Hour/Weekend Analysis

In [ ]:
# Categorical count plots
for feat in ["gh_is_pr"]:
    if feat in sdf.columns:
        _tmp = sdf.copy()
        _tmp["Outcome"] = _tmp["y_raw"].map({0:"Passed",1:"Failed"})
        fig, axes = plt.subplots(1,2,figsize=(12,4))
        fig.suptitle(f"{feat} vs Build Outcome", fontweight="bold",fontsize=12)
        sns.countplot(data=_tmp, x=feat, hue="Outcome",
                      palette={"Passed":"#43A047","Failed":"#E53935"}, ax=axes[0])
        axes[0].set_title("Count", fontweight="bold"); axes[0].grid(axis="y",alpha=0.3)
        fr = _tmp.groupby(feat)["y_raw"].mean()*100
        axes[1].bar(fr.index.astype(str), fr.values,
                    color=["#42A5F5","#EF5350"], edgecolor="white")
        axes[1].set_title("Failure Rate %", fontweight="bold")
        axes[1].set_ylabel("Failure %"); axes[1].grid(axis="y",alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"vis_count_{feat}.png", dpi=130, bbox_inches="tight"); plt.show()

# Off-hour & weekend
if TIMESTAMP in sdf.columns:
    sdf["_hour"] = sdf[TIMESTAMP].dt.hour
    sdf["_dow"]  = sdf[TIMESTAMP].dt.dayofweek
    sdf["_off_hour"] = ((sdf["_hour"]>=22)|(sdf["_hour"]<6)).astype(int)
    sdf["_weekend"]  = (sdf["_dow"]>=5).astype(int)

    fig, axes = plt.subplots(1,2,figsize=(12,4))
    fig.suptitle("Off-Hour & Weekend vs Failure Rate",fontweight="bold",fontsize=12)
    for ax, col, label in zip(axes,["_off_hour","_weekend"],
                               ["Off-Hour (22-06)","Weekend"]):
        fr = sdf.groupby(col)["y_raw"].mean()*100
        bars = ax.bar(["No","Yes"], fr.values,
                      color=["#42A5F5","#EF5350"], edgecolor="white", width=0.45)
        ax.set_ylabel("Failure Rate (%)"); ax.set_title(label,fontweight="bold")
        ax.grid(axis="y",alpha=0.3)
        for bar,v in zip(bars, fr.values):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.3,
                    f"{v:.1f}%", ha="center", fontsize=12)
    plt.tight_layout()
    plt.savefig("vis_temporal_flags.png", dpi=130, bbox_inches="tight"); plt.show()


### 3E — Top Branches Failure Rate

In [ ]:
if "git_branch" in sdf.columns:
    top_b  = sdf["git_branch"].value_counts().head(10).index
    _tmp   = sdf[sdf["git_branch"].isin(top_b)].copy()
    br_fr  = _tmp.groupby("git_branch")["y_raw"].mean().sort_values()*100
    fig, ax = plt.subplots(figsize=(10,5))
    br_fr.plot.barh(ax=ax, color="#EF5350", edgecolor="white")
    ax.set_title("Failure Rate by Branch (top 10 by frequency)",
                 fontweight="bold", fontsize=12)
    ax.set_xlabel("Failure Rate (%)"); ax.grid(axis="x",alpha=0.3)
    for bar,v in zip(ax.patches, br_fr.values):
        ax.text(v+0.3, bar.get_y()+bar.get_height()/2,
                f"{v:.1f}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("vis_branch_failure.png", dpi=130, bbox_inches="tight"); plt.show()


### 3F — Outlier Boxplots (ero-Inflation Aware IQR)

>  Q1=Q3=0 for columns like `git_diff_src_churn` (>75% zeros).
> IQR=0 made every non-zero value look like an outlier.
>
> IQR computed on the **non-zero subset**. Zero-inflation % is
> reported separately.


In [ ]:
KEY_NUM = ["git_diff_src_churn","gh_sloc","gh_team_size",
           "gh_num_pr_comments","gh_test_cases_per_kloc"]
key_num = [c for c in KEY_NUM if c in sdf.columns and c not in ZERO_VAR_COLS]

print("── IQR Outlier Summary (non-zero subset) ──")
print(f"{'Column':<28}{'Zero%':>7}{'Q1':>9}{'Q3':>9}{'IQR':>9}"
      f"{'Lower':>10}{'Upper':>10}{'Outliers':>10}{'Out%':>7}")
print("-"*95)
for col in key_num:
    _col_all = sdf[col].dropna()
    zero_pct = (_col_all==0).mean()*100
    _col_nz  = _col_all[_col_all>0]          # non-zero subset
    if len(_col_nz) < 100:
        print(f"{col:<28}  {zero_pct:>5.1f}%  (insufficient non-zero values)")
        continue
    q1,q3 = _col_nz.quantile([0.25,0.75])
    iqr   = q3-q1
    lo,hi = q1-1.5*iqr, q3+1.5*iqr
    n_out = ((_col_nz<lo)|(_col_nz>hi)).sum()
    out_pct = n_out/len(_col_nz)*100
    print(f"{col:<28}{zero_pct:>6.1f}%{q1:>9.1f}{q3:>9.1f}{iqr:>9.1f}"
          f"{lo:>10.1f}{hi:>10.1f}{n_out:>10,}{out_pct:>6.2f}%")

n = len(key_num)
fig, axes = plt.subplots(1,n,figsize=(3.3*n,5))
if n==1: axes=[axes]
fig.suptitle("Outlier Boxplots — log10 scale (50k sample)",
             fontweight="bold",fontsize=12)
for ax, col in zip(axes, key_num):
    _x = np.log10(sdf[col].dropna().clip(lower=0)+1)
    ax.boxplot(_x, vert=True, patch_artist=True,
               boxprops=dict(facecolor="#90CAF9",color="#1565C0"),
               medianprops=dict(color="#E53935",linewidth=2.5),
               whiskerprops=dict(color="#1565C0"),
               flierprops=dict(marker=".",alpha=0.15,markersize=2))
    zero_pct = (sdf[col].fillna(0)==0).mean()*100
    ax.set_title(f"{col}\n(zero={zero_pct:.0f}%)",fontsize=7,fontweight="bold")
    ax.set_xticks([]); ax.set_ylabel("log10(x+1)"); ax.grid(axis="y",alpha=0.3)
plt.tight_layout()
plt.savefig("vis_outlier_boxplots.png", dpi=130, bbox_inches="tight"); plt.show()


### 3G — Pair Plot (10k sample)

In [ ]:
PAIR_FEATS = ["git_diff_src_churn","gh_sloc","gh_team_size","gh_num_pr_comments"]
pair_feats = [f for f in PAIR_FEATS if f in sdf.columns and f not in ZERO_VAR_COLS]

_pp = sdf[pair_feats+["y_raw"]].copy()
for f in pair_feats:
    _pp[f] = np.log10(_pp[f].clip(lower=0)+1)
_pp["Outcome"] = _pp["y_raw"].map({0:"Passed",1:"Failed"})
_pp_s = _pp.sample(min(PAIR_N,len(_pp)), random_state=42)

g = sns.pairplot(_pp_s.drop(columns=["y_raw"]), hue="Outcome",
                 palette={"Passed":"#43A047","Failed":"#E53935"},
                 plot_kws={"alpha":0.2,"s":7}, diag_kind="kde")
g.figure.suptitle("Pair Plot — Key Features (10k sample, log-scaled)",
                   y=1.01, fontweight="bold", fontsize=12)
plt.savefig("vis_pairplot.png", dpi=110, bbox_inches="tight"); plt.show()
del _pp, _pp_s; gc.collect()


### 3H — Failure Rate by Hour & Day

In [ ]:
if "_hour" in sdf.columns:
    fig, axes = plt.subplots(1,2,figsize=(14,4))
    fig.suptitle("Failure Rate by Hour and Day of Week (50k sample)",
                 fontweight="bold",fontsize=13)
    hr_rate = sdf.groupby("_hour")["y_raw"].mean()*100
    axes[0].bar(hr_rate.index, hr_rate.values, color="#EF5350", edgecolor="white")
    axes[0].axhline(hr_rate.mean(), ls="--", color="#1565C0", lw=1.5, label="Mean")
    axes[0].set_xlabel("Hour (UTC)"); axes[0].set_ylabel("Failure %")
    axes[0].set_title("By Hour", fontweight="bold")
    axes[0].legend(); axes[0].grid(axis="y",alpha=0.3)

    day_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    day_rate  = sdf.groupby("_dow")["y_raw"].mean()*100
    day_rate.index = [day_names[i] for i in day_rate.index]
    bar_colors = ["#EF5350" if d in ["Sat","Sun"] else "#42A5F5"
                  for d in day_rate.index]
    axes[1].bar(day_rate.index, day_rate.values,
                color=bar_colors, edgecolor="white")
    axes[1].axhline(day_rate.mean(), ls="--", color="#1565C0", lw=1.5, label="Mean")
    axes[1].set_ylabel("Failure %"); axes[1].set_title("By Day (Red=Weekend)",
                                                        fontweight="bold")
    axes[1].legend(); axes[1].grid(axis="y",alpha=0.3)
    plt.tight_layout()
    plt.savefig("vis_temporal_failure_rate.png", dpi=130, bbox_inches="tight"); plt.show()

del sdf; gc.collect()
mem_check("post Phase 3")


---
## Phase 4 — Preprocessing

`drop_duplicates()` uses business-key subset only.
Also drops: `tr_log_*`, zero-variance columns, 100%-null columns,
`tr_duration`, `tr_jobs`, `tr_job_id`, `tr_testduration`.


In [ ]:
# ── 4. Preprocessing (smart dedup) ────────────────────────────────────

# Columns to explicitly drop
DROP_EXPLICIT = (
    [c for c in raw_df.columns if c.startswith("tr_log_")] +
    ["tr_duration","tr_jobs","tr_job_id","tr_testduration"]
)

# Auto-drop zero-variance numeric columns (found in 2H)
DROP_ZV = ZERO_VAR_COLS.copy()

# Auto-drop columns with >=99% missing (found in 2F)
_null_pct = raw_df.isna().mean()*100
DROP_NULL99 = _null_pct[_null_pct>=99.0].index.tolist()

ALL_DROP = list(set(DROP_EXPLICIT + DROP_ZV + DROP_NULL99))
print(f"Explicit drops   : {len(DROP_EXPLICIT)}")
print(f"Zero-var drops   : {len(DROP_ZV)}  → {DROP_ZV}")
print(f"≥99% null drops  : {len(DROP_NULL99)} → {DROP_NULL99}")
print(f"Total drops      : {len(ALL_DROP)}")

keep = [c for c in raw_df.columns if c not in ALL_DROP]
df   = raw_df[keep].copy()
del raw_df; gc.collect()

# ── smart dedup on business key ───────────────────────────────────
n_bef = len(df)
dedup_cols_avail = [c for c in DEDUP_SUBSET if c in df.columns]
if dedup_cols_avail:
    df.drop_duplicates(subset=dedup_cols_avail, inplace=True)
    print(f"\nBusiness-key dedup removed : {n_bef-len(df):,} rows")
else:
    # Fallback: full-row dedup if key cols missing
    df.drop_duplicates(inplace=True)
    print(f"\nFull-row dedup removed : {n_bef-len(df):,} rows")
df.reset_index(drop=True, inplace=True)

# Filter canceled & encode target
df = df[df[TARGET]!="canceled"].copy()
df["y"] = df[TARGET].isin(["failed","errored"]).astype("int8")
df.drop(columns=[TARGET], inplace=True)

# Parse timestamp & sort
if not pd.api.types.is_datetime64_any_dtype(df[TIMESTAMP]):
    df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], errors="coerce")
df.sort_values(TIMESTAMP, inplace=True)
df.reset_index(drop=True, inplace=True)

# Final downcast
for col in df.select_dtypes("float64").columns:
    df[col] = df[col].astype("float32")
for col in df.select_dtypes("int64").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

mem_check("preprocessing complete")
print(f"\nClean shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Failure rate : {df['y'].mean():.4%}")
print(f"Date range   : {df[TIMESTAMP].min().date()} → {df[TIMESTAMP].max().date()}")

# Record cleaned numeric columns for correlation in Phase 5
clean_num_cols = df.select_dtypes(include=[np.number]).columns.tolist()


---
## Phase 5 — Post-Preprocessing Correlation & Distribution Shift

- Correlation now runs on the **cleaned df** (after `tr_log_*` removal),
so leaky log-columns no longer dominate the heatmap.

- Temporal distribution shift (train vs val vs test failure rate)
is explicitly measured and visualised.


### 5A — Pearson Correlation Heatmap (Cleaned Features)

In [ ]:
# ── 5A. Correlation on clean df ──────────────────────────────────────────────
_sc = df[clean_num_cols].sample(min(100_000,len(df)), random_state=42)
corr_clean = _sc.corr(method="pearson")
del _sc; gc.collect()

fig_w = max(14, len(corr_clean)*0.5)
fig, ax = plt.subplots(figsize=(fig_w, fig_w*0.85))
mask = np.triu(np.ones_like(corr_clean, dtype=bool))
sns.heatmap(corr_clean, mask=mask, cmap="coolwarm", center=0,
            linewidths=0.3, annot=(len(corr_clean)<=18),
            fmt=".2f", square=True, ax=ax, cbar_kws={"shrink":0.6},
            annot_kws={"size":7})
ax.set_title("Pearson Correlation — Cleaned Features (no tr_log_*)",
             fontweight="bold", fontsize=13)
plt.xticks(fontsize=7, rotation=45, ha="right")
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig("clean_correlation_heatmap.png", dpi=130, bbox_inches="tight"); plt.show()

# Top pairs
pairs = (corr_clean.where(np.tril(np.ones(corr_clean.shape),k=-1).astype(bool))
                   .stack().reset_index())
pairs.columns = ["A","B","r"]
pairs["|r|"] = pairs["r"].abs()
print("── Top 15 correlated pairs (clean df) ──")
print(pairs.sort_values("|r|",ascending=False).head(15)[["A","B","r"]].to_string(index=False))


### 5B — Feature → Target Correlation Bar (Cleaned)

In [ ]:
if "y" in corr_clean.columns:
    target_corr = corr_clean["y"].drop("y").sort_values()
    fig, ax = plt.subplots(figsize=(8, max(5,len(target_corr)*0.22)))
    colors = ["#E53935" if v>0 else "#43A047" for v in target_corr.values]
    ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor="white")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title("Feature Correlation with Target y (cleaned features)",
                 fontweight="bold",fontsize=12)
    ax.set_xlabel("Pearson r"); ax.grid(axis="x",alpha=0.3)
    plt.tight_layout()
    plt.savefig("clean_feature_target_corr.png", dpi=130, bbox_inches="tight"); plt.show()
    print(target_corr.sort_values(ascending=False).to_string())


### 5C — Temporal Distribution Shift


In [ ]:
# ── 5C. Temporal distribution shift ──────────────────────────────────
N  = len(df)
t1 = int(N*0.70); t2 = int(N*0.85)

splits = {
    "Train (70%)"      : df.iloc[:t1],
    "Validation (15%)" : df.iloc[t1:t2],
    "Test (15%)"       : df.iloc[t2:],
}

print("── Temporal Distribution Shift ──")
print(f"{'Split':<20} {'Rows':>9}  {'Fail%':>8}  {'Date range'}")
print("-"*70)
fail_rates = {}
for name, part in splits.items():
    dr = f"{part[TIMESTAMP].min().date()} → {part[TIMESTAMP].max().date()}"
    fr = part["y"].mean()*100
    fail_rates[name] = fr
    print(f"{name:<20} {len(part):>9,}  {fr:>7.2f}%  {dr}")

# Visualise monthly failure rate with split boundaries
monthly2 = (df.set_index(TIMESTAMP).resample("ME")["y"]
              .agg(total="count", failures="sum"))
monthly2["fail_rate"] = (monthly2["failures"]/monthly2["total"]*100).round(2)
split_dates = [df.iloc[t1][TIMESTAMP], df.iloc[t2][TIMESTAMP]]

fig, ax = plt.subplots(figsize=(14,4))
ax.plot(monthly2.index, monthly2["fail_rate"],
        color="#E53935", lw=2, marker="o", ms=4)
ax.axvline(split_dates[0], color="#1565C0", ls="--", lw=2, label="Train/Val split")
ax.axvline(split_dates[1], color="#2E7D32", ls="--", lw=2, label="Val/Test split")
ax.fill_between(monthly2.index, monthly2["fail_rate"],
                alpha=0.08, color="#E53935")
ax.set_xlabel("Month"); ax.set_ylabel("Monthly Failure Rate (%)")
ax.set_title("Temporal Distribution Shift — Failure Rate Over Time",
             fontweight="bold", fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("temporal_shift.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"\nTrain→Val failure rate drop: "
      f"{fail_rates['Train (70%)']:.2f}% → {fail_rates['Validation (15%)']:.2f}% "
      f"(Δ={fail_rates['Train (70%)']-fail_rates['Validation (15%)']:.2f}%)")
print("   This temporal drift is normal for chronological splits.")
print("   XGBoost scale_pos_weight is computed from TRAIN set only.")


---
## Phase 6 — XGBoost Feature Engineering

`gh_num_commits_in_push` is 100% null in this dataset → skipped.
Any feature derived from a zero-variance column is guarded with
a null-check and skipped rather than producing all-zero columns.


In [ ]:
# ── 6. Feature engineering ────────────────────────────────────────
mem_check("pre feature eng")
f32 = lambda s: s.astype("float32")
i8  = lambda s: s.astype("int8")
ts  = df[TIMESTAMP]

def col_ok(name: str, min_nonzero_pct: float = 1.0) -> bool:
    """Return True if column exists, is not zero-variance,
    and has at least min_nonzero_pct % non-zero values."""
    if name not in df.columns:        return False
    if name in ZERO_VAR_COLS:         return False
    nz = (df[name].fillna(0) != 0).mean()*100
    if nz < min_nonzero_pct:          return False
    return True

# ── A: Author behaviour ───────────────────────────────────────────────────
df["author_commit_span_days"] = f32(
    df.groupby("gh_project_name")[TIMESTAMP]
      .transform(lambda x: (x-x.min()).dt.total_seconds()/86_400))

df["author_historical_fail_rate"] = f32(
    df.groupby("gh_project_name")["y"]
      .transform(lambda x: x.astype("float32").shift(1)
                             .rolling(30,min_periods=1).mean()))

df["author_build_count_30d"] = (
    df.groupby("gh_project_name")["y"]
      .transform(lambda x: x.shift(1).rolling(30,min_periods=1).count())
      .astype("int16"))

_np = df.groupby("gh_project_name")["gh_project_name"].transform("nunique").astype("int16")
df["is_multi_project_author"] = i8((_np>1).astype("int8"))
df["repo_switch_velocity"]    = f32(_np.astype("float32")/(df["author_commit_span_days"]+1))
del _np; gc.collect()

# ── B: PR quality ─────────────────────────────────────────────────────────
_src  = df.get("git_diff_src_churn", pd.Series(0.0,index=df.index)).fillna(0).astype("float32")
_prc  = df.get("gh_num_pr_comments", pd.Series(0.0,index=df.index)).fillna(0).astype("float32")
_ispr = df.get("gh_is_pr", pd.Series(False,index=df.index)).fillna(False).astype(bool)
df["is_unreviewed_pr_merge"] = i8((_ispr&(_prc==0)&(_src>300)).astype("int8"))
df["pr_comment_density"]     = f32(_prc/(_src+1))
del _ispr, _prc; gc.collect()

# ── C: Temporal / cyclic ──────────────────────────────────────────────────
_hour = ts.dt.hour.astype("int8")
_dow  = ts.dt.dayofweek.astype("int8")
df["is_off_hour_commit"] = i8(((_hour>=22)|(_hour<6)).astype("int8"))
df["is_weekend_commit"]  = i8((_dow>=5).astype("int8"))
df["hour_sin"]           = f32(np.sin(2*np.pi*_hour/24).astype("float32"))
df["hour_cos"]           = f32(np.cos(2*np.pi*_hour/24).astype("float32"))
del _hour, _dow; gc.collect()

# ── D: Code-churn ratios (guard zero-variance test churn) ─────────
_sloc = df.get("gh_sloc", pd.Series(1.0,index=df.index)).fillna(1).replace(0,1).astype("float32")
df["churn_ratio"]            = f32(_src/_sloc)
df["log_git_diff_src_churn"] = f32(np.log10(_src.clip(lower=0)+1))

if col_ok("git_diff_test_churn"):               # only if not zero-variance
    _test = df["git_diff_test_churn"].fillna(0).astype("float32")
    df["log_git_diff_test_churn"] = f32(np.log10(_test.clip(lower=0)+1))
    df["test_coverage_ratio"]     = f32(_test/(_src+1))
    del _test
else:
    print("git_diff_test_churn is zero-variance — skipping derived features")
    df["log_git_diff_test_churn"] = np.float32(0)
    df["test_coverage_ratio"]     = np.float32(0)
gc.collect()

# ── E: Test quality & branch ──────────────────────────────────────────────
df["test_density_kloc"]  = f32(df.get("gh_test_cases_per_kloc",
                                        pd.Series(0.0,index=df.index)).fillna(0))
df["test_lines_deleted"] = f32(df.get("gh_diff_tests_deleted",
                                       pd.Series(0.0,index=df.index)).fillna(0))
MAIN_BRANCHES = {"main","master","production","release"}
if "git_branch" in df.columns:
    df["is_main_branch"] = i8(df["git_branch"].str.lower().isin(MAIN_BRANCHES).astype("int8"))
    _bcc = (df.groupby(["gh_project_name","git_branch"])
              ["git_trigger_commit"].transform("nunique").astype("int16"))
    df["branch_contributor_count"] = _bcc; del _bcc
else:
    df["is_main_branch"]           = np.int8(0)
    df["branch_contributor_count"] = np.int16(1)
gc.collect()

# ── F: Push & codebase scale (guard 100%-null cols) ───────────────
if col_ok("gh_num_commits_in_push"):             # skip if all null
    df["commits_in_push"] = df["gh_num_commits_in_push"].fillna(1).astype("int16")
else:
    print("gh_num_commits_in_push is null — commits_in_push omitted")
    df["commits_in_push"] = np.int16(1)

df["log_gh_sloc"]      = f32(np.log10(_sloc.clip(lower=0)+1))
df["log_gh_team_size"] = f32(np.log10(
    df.get("gh_team_size",pd.Series(1.0,index=df.index)).fillna(1).clip(lower=0)+1))
del _sloc, _src; gc.collect()

XGB_FEATS = [c for c in [
    "author_commit_span_days","author_historical_fail_rate","author_build_count_30d",
    "is_multi_project_author","repo_switch_velocity","is_unreviewed_pr_merge",
    "pr_comment_density","is_off_hour_commit","is_weekend_commit",
    "hour_sin","hour_cos","churn_ratio","log_git_diff_src_churn",
    "log_git_diff_test_churn","test_coverage_ratio","test_density_kloc",
    "test_lines_deleted","is_main_branch","branch_contributor_count",
    "commits_in_push","log_gh_sloc","log_gh_team_size",
] if c in df.columns]

mem_check("feature engineering done")
print(f"\nFinal XGBoost feature set: {len(XGB_FEATS)} features")
for i,f in enumerate(XGB_FEATS,1):
    nz = (df[f].fillna(0)!=0).mean()*100
    print(f"  {i:2d}. {f:<35}  non-zero: {nz:5.1f}%")


---
## Phase 7 — Post-Feature-Engineering Insights

In [ ]:
# ── 7A. Engineered feature describe ─────────────────────────────────────────
feat_df = df[XGB_FEATS+["y"]]
desc_fe = feat_df[XGB_FEATS].describe(percentiles=[.05,.25,.50,.75,.95]).T
desc_fe["skewness"] = feat_df[XGB_FEATS].skew().round(3)
desc_fe["kurtosis"] = feat_df[XGB_FEATS].kurtosis().round(3)
desc_fe["zero_pct"] = ((feat_df[XGB_FEATS]==0).mean()*100).round(2)
pd.set_option("display.max_columns",14); pd.set_option("display.width",280)
print("── Engineered feature descriptive statistics ──")
print(desc_fe.to_string())


In [ ]:
# ── 7B. Engineered feature correlation heatmap ───────────────────────────────
_sfe    = feat_df.sample(min(100_000,len(feat_df)), random_state=42)
corr_fe = _sfe.corr(method="pearson")
del _sfe; gc.collect()

fig, ax = plt.subplots(figsize=(15,13))
mask = np.triu(np.ones_like(corr_fe, dtype=bool))
sns.heatmap(corr_fe, mask=mask, cmap="coolwarm", center=0,
            linewidths=0.4, annot=True, fmt=".2f", square=True, ax=ax,
            cbar_kws={"shrink":0.6}, annot_kws={"size":7})
ax.set_title("Engineered Feature Correlation Matrix",
             fontweight="bold", fontsize=13)
plt.xticks(fontsize=8,rotation=45,ha="right"); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig("fe_correlation_heatmap.png", dpi=130, bbox_inches="tight"); plt.show()

pairs_fe = (corr_fe.where(np.tril(np.ones(corr_fe.shape),k=-1).astype(bool))
                   .stack().reset_index())
pairs_fe.columns=["A","B","r"]; pairs_fe["|r|"]=pairs_fe["r"].abs()
print("── Top 10 engineered feature pairs ──")
print(pairs_fe.sort_values("|r|",ascending=False).head(10)[["A","B","r"]].to_string(index=False))


In [ ]:
# ── 7C. Violin plots — engineered features vs target ─────────────────────────
_sfe2 = feat_df.sample(min(SAMPLE_N,len(feat_df)), random_state=99).copy()
_sfe2["Outcome"] = _sfe2["y"].map({0:"Passed",1:"Failed"})

fig, axes = plt.subplots(4,6,figsize=(24,16))
axes = axes.flatten()
fig.suptitle("Engineered Features — Violin Plots vs Outcome (50k sample)",
             fontweight="bold",fontsize=14)
for ax, feat in zip(axes, XGB_FEATS):
    sns.violinplot(data=_sfe2, x="Outcome", y=feat,
                   palette={"Passed":"#43A047","Failed":"#E53935"},
                   inner="box", cut=0, ax=ax, linewidth=0.8)
    ax.set_title(feat, fontsize=7, fontweight="bold")
    ax.set_xlabel(""); ax.grid(axis="y",alpha=0.3)
for ax in axes[len(XGB_FEATS):]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig("fe_violin_plots.png", dpi=110, bbox_inches="tight"); plt.show()
del _sfe2; gc.collect()


In [ ]:
# ── 7D. Feature → target correlation ranking ─────────────────────────────────
fe_target_corr = corr_fe["y"].drop("y").sort_values()
fig, ax = plt.subplots(figsize=(8,7))
colors = ["#E53935" if v>0 else "#43A047" for v in fe_target_corr.values]
ax.barh(fe_target_corr.index, fe_target_corr.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Engineered Feature Correlation with Target y",
             fontweight="bold",fontsize=12)
ax.set_xlabel("Pearson r"); ax.grid(axis="x",alpha=0.3)
plt.tight_layout()
plt.savefig("fe_target_correlation.png", dpi=130, bbox_inches="tight"); plt.show()
print(fe_target_corr.sort_values(ascending=False).to_string())


---
## Phase 8 — Chronological Split (70 / 15 / 15)

In [ ]:
N  = len(df); t1 = int(N*0.70); t2 = int(N*0.85)

train_df = df.iloc[:t1]; val_df = df.iloc[t1:t2]; test_df = df.iloc[t2:]

print(f"{'Split':<14}{'Rows':>9}  {'Fail%':>8}  Date range")
print("-"*65)
for name, part in [("Train",train_df),("Validation",val_df),("Test",test_df)]:
    dr = f"{part[TIMESTAMP].min().date()} → {part[TIMESTAMP].max().date()}"
    print(f"{name:<14}{len(part):>9,}  {part['y'].mean():>7.2%}  {dr}")

X_tr = train_df[XGB_FEATS].to_numpy("float32"); y_tr = train_df["y"].to_numpy("int8")
X_va = val_df[XGB_FEATS].to_numpy("float32");   y_va = val_df["y"].to_numpy("int8")
X_te = test_df[XGB_FEATS].to_numpy("float32");  y_te = test_df["y"].to_numpy("int8")

# Store test split index for aligned comparison
TEST_START_IDX = t2
mem_check("post split")


---
## Phase 9 — XGBoost Training (Independent)

Optimal decision threshold auto-selected from validation set
(maximise F1 on Failed class) instead of hardcoded 0.50.


### 9A — Train XGBoost

In [ ]:
spw = float((y_tr==0).sum()) / float((y_tr==1).sum()+1e-9)
print(f"scale_pos_weight (exact) = {spw:.4f}")

xgb_model = XGBClassifier(
    n_estimators          = 2000,
    learning_rate         = 0.05,
    max_depth             = 6,
    min_child_weight      = 3,
    subsample             = 0.80,
    colsample_bytree      = 0.80,
    scale_pos_weight      = spw,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    gamma                 = 0.05,
    tree_method           = "hist",
    eval_metric           = ["logloss","auc"],
    n_jobs                = -1,
    random_state          = 42,
    early_stopping_rounds = 50,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_tr,y_tr),(X_va,y_va)], verbose=100)
print(f"\nBest iteration : {xgb_model.best_iteration}")
print(f"Best val score : {xgb_model.best_score:.4f}")
mem_check("post XGBoost training")


### 9B — XGBoost Learning Curves (Per Round)

In [ ]:
evals  = xgb_model.evals_result()
tr_loss = evals["validation_0"]["logloss"]
va_loss = evals["validation_1"]["logloss"]
tr_auc  = evals["validation_0"]["auc"]
va_auc  = evals["validation_1"]["auc"]
rounds  = list(range(1, len(tr_loss)+1))
best_r  = xgb_model.best_iteration+1

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle("XGBoost — Training Curves (per Boosting Round)",
             fontweight="bold",fontsize=13)

axes[0].plot(rounds, tr_loss, lw=1.8, color="#42A5F5", label="Train Log-Loss")
axes[0].plot(rounds, va_loss, lw=1.8, color="#EF5350", ls="--", label="Val Log-Loss")
axes[0].axvline(best_r, color="green", ls=":", lw=2, label=f"Best ({best_r})")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("Log-Loss")
axes[0].set_title("Log-Loss", fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rounds, tr_auc, lw=1.8, color="#42A5F5", label="Train AUC")
axes[1].plot(rounds, va_auc, lw=1.8, color="#EF5350", ls="--", label="Val AUC")
axes[1].axvline(best_r, color="green", ls=":", lw=2, label=f"Best ({best_r})")
axes[1].set_xlabel("Round"); axes[1].set_ylabel("AUC")
axes[1].set_title("AUC-ROC", fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("xgb_learning_curves.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"Best round val — loss: {va_loss[best_r-1]:.4f}  AUC: {va_auc[best_r-1]:.4f}")


### 9C — Optimal Threshold Selection + Evaluation

In [ ]:
# ── FIX 8: auto-select threshold on val set maximising Failed F1 ─────────────
xgb_proba_va = xgb_model.predict_proba(X_va)[:,1].astype("float32")
best_f1, best_thr = 0.0, 0.50
for thr in np.arange(0.20, 0.71, 0.01):
    pred = (xgb_proba_va >= thr).astype("int8")
    f1   = f1_score(y_va, pred, pos_label=1, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"Optimal threshold (val F1-Failed): {best_thr:.2f}  "
      f"→ val F1={best_f1:.4f}  (default 0.50 gave "
      f"{f1_score(y_va,(xgb_proba_va>=0.50).astype('int8'),pos_label=1):.4f})")

XGB_THRESHOLD = best_thr

# Evaluate on test set
xgb_proba_te = xgb_model.predict_proba(X_te)[:,1].astype("float32")
xgb_pred_te  = (xgb_proba_te >= XGB_THRESHOLD).astype("int8")

print("\n" + "━"*62)
print(f"  XGBoost — Test Set (threshold={XGB_THRESHOLD:.2f})")
print("━"*62)
print(classification_report(y_te, xgb_pred_te,
                             target_names=["Passed","Failed"], digits=4))
xgb_auc = roc_auc_score(y_te, xgb_proba_te)
xgb_mcc = matthews_corrcoef(y_te, xgb_pred_te)
xgb_ll  = log_loss(y_te, xgb_proba_te)
print(f"  AUC-ROC  : {xgb_auc:.4f}")
print(f"  MCC      : {xgb_mcc:.4f}")
print(f"  Log-Loss : {xgb_ll:.4f}")

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(
    y_te, xgb_pred_te, display_labels=["Passed","Failed"],
    cmap="Blues", ax=ax)
ax.set_title(f"XGBoost Confusion Matrix (thr={XGB_THRESHOLD:.2f})",
             fontweight="bold")
plt.tight_layout()
plt.savefig("xgb_confusion_matrix.png", dpi=130, bbox_inches="tight"); plt.show()


### 9D — ROC + Precision-Recall Curves

In [ ]:
fpr,tpr,_  = roc_curve(y_te, xgb_proba_te)
prec,rec,_ = precision_recall_curve(y_te, xgb_proba_te)
ap         = average_precision_score(y_te, xgb_proba_te)

fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("XGBoost — ROC & Precision-Recall Curves",
             fontweight="bold",fontsize=13)
axes[0].plot(fpr,tpr,lw=2,color="#1565C0",label=f"AUC={xgb_auc:.4f}")
axes[0].plot([0,1],[0,1],ls="--",color="gray",lw=1)
axes[0].fill_between(fpr,tpr,alpha=0.08,color="#1565C0")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve",fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec,prec,lw=2,color="#E53935",label=f"AP={ap:.4f}")
axes[1].axhline(y_te.mean(),ls="--",color="gray",lw=1,
                label=f"Baseline ({y_te.mean():.2%})")
axes[1].fill_between(rec,prec,alpha=0.08,color="#E53935")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve",fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("xgb_roc_pr.png", dpi=130, bbox_inches="tight"); plt.show()


### 9E — Threshold Sweep Analysis

In [ ]:
rows = []
for thr in np.arange(0.10, 0.91, 0.05):
    pred = (xgb_proba_te>=thr).astype("int8")
    rows.append({
        "Thr"  : round(thr,2),
        "Prec" : round(precision_score(y_te,pred,zero_division=0),4),
        "Rec"  : round(recall_score(y_te,pred,zero_division=0),4),
        "F1"   : round(f1_score(y_te,pred,zero_division=0),4),
        "MCC"  : round(matthews_corrcoef(y_te,pred),4),
        "Acc"  : round(accuracy_score(y_te,pred),4),
    })
thr_df = pd.DataFrame(rows)
print("── XGBoost Threshold Sweep ──")
print(thr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10,4))
for col, marker in [("Prec","o"),("Rec","s"),("F1","^"),("MCC","D")]:
    ax.plot(thr_df["Thr"], thr_df[col], label=col, lw=2, marker=marker, ms=4)
ax.axvline(XGB_THRESHOLD, color="green", ls="--", lw=2,
           label=f"Optimal thr={XGB_THRESHOLD:.2f}")
ax.axvline(0.50, color="gray", ls=":", lw=1.5, label="Default 0.50")
ax.set_xlabel("Decision Threshold"); ax.set_ylabel("Score")
ax.set_title("XGBoost — Threshold vs Metrics",fontweight="bold",fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("xgb_threshold_analysis.png", dpi=130, bbox_inches="tight"); plt.show()


### 9F — Feature Importance (Gain / Cover / Weight)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(21,7))
fig.suptitle("XGBoost Feature Importance — Gain / Cover / Weight",
             fontweight="bold",fontsize=13)
for ax, imp_type, col in zip(axes,["gain","cover","weight"],
                              ["#1565C0","#2E7D32","#6A1B9A"]):
    scores = xgb_model.get_booster().get_score(importance_type=imp_type)
    fi     = pd.Series(scores).reindex(XGB_FEATS).fillna(0).sort_values()
    fi.tail(20).plot.barh(ax=ax, color=col, edgecolor="white")
    ax.set_title(f"Importance: {imp_type}",fontweight="bold",fontsize=11)
    ax.set_xlabel(imp_type.capitalize()); ax.grid(axis="x",alpha=0.3)
plt.tight_layout()
plt.savefig("xgb_feature_importance.png", dpi=130, bbox_inches="tight"); plt.show()


In [ ]:
xgb_model.save_model("aiops_xgboost_model.json")
print("XGBoost → aiops_xgboost_model.json")
mem_check("XGBoost complete")


---
## Phase 10 — LSTM Sequence Engineering

In [ ]:
WINDOW     = 10
MIN_BUILDS = 10
SEQ_FEATS  = [
    "consecutive_failures","fail_streak_count","rolling_fail_rate_5",
    "time_gap_seconds","time_since_last_failure",
    "log_git_diff_src_churn","churn_velocity","author_switch_flag",
]

def build_seq_features(grp: pd.DataFrame) -> pd.DataFrame:
    grp = grp.sort_values(TIMESTAMP).reset_index(drop=True)
    y_s = grp["y"].astype("float32")
    grp["consecutive_failures"]    = y_s.shift(1).fillna(0).astype("float32")
    _sr = (y_s.shift(1).fillna(0)==0).astype(int).cumsum()
    grp["fail_streak_count"]       = (y_s.shift(1).fillna(0)
                                        .groupby(_sr).cumsum().astype("float32"))
    grp["rolling_fail_rate_5"]     = (y_s.shift(1).rolling(5,min_periods=1)
                                        .mean().fillna(0).astype("float32"))
    _ts = grp[TIMESTAMP].astype("int64")//1_000_000_000
    grp["time_gap_seconds"]        = _ts.diff().fillna(0).clip(lower=0).astype("float32")
    _ft = _ts.where(y_s.shift(1)==1)
    grp["time_since_last_failure"] = (_ts-_ft.ffill().fillna(_ts.iloc[0])
                                       ).clip(lower=0).astype("float32")
    # guard zero-variance churn (FIX 5)
    if "git_diff_src_churn" in grp.columns and grp["git_diff_src_churn"].var() > 0:
        _ch = grp["git_diff_src_churn"].fillna(0)
    else:
        _ch = pd.Series(0.0, index=grp.index)
    grp["log_git_diff_src_churn"]  = np.log10(_ch.clip(lower=0)+1).astype("float32")
    grp["churn_velocity"]          = _ch.diff().fillna(0).astype("float32")
    _a = "git_trigger_commit"
    grp["author_switch_flag"] = ((grp[_a]!=grp[_a].shift(1)).astype("float32")
                                  if _a in grp.columns else np.float32(0))
    return grp[SEQ_FEATS+["y"]]

print("Building per-repo sequences …")
frames, skipped = [], 0
for repo, grp in df.groupby("gh_project_name", sort=False):
    if len(grp) >= MIN_BUILDS:
        frames.append(build_seq_features(grp))
    else:
        skipped += 1

print(f"Repos used    : {len(frames):,}  |  skipped: {skipped:,}")
seq_df = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

N_seq = len(seq_df)
s_tr  = int(N_seq*0.70); s_va = int(N_seq*0.85)

scaler  = StandardScaler()
_X_raw  = seq_df[SEQ_FEATS].values.astype("float32")
scaler.fit(_X_raw[:s_tr])
_X_sc   = scaler.transform(_X_raw).astype("float32")
_y_arr  = seq_df["y"].values.astype("int8")
del _X_raw, seq_df; gc.collect()

joblib.dump(scaler, "aiops_lstm_scaler.pkl")
print("Scaler → aiops_lstm_scaler.pkl")

def make_tensors(X, y, w):
    N,F   = X.shape
    Xo    = np.zeros((N-w,w,F), dtype="float32")
    yo    = np.empty(N-w, dtype="int8")
    for i in range(w,N):
        Xo[i-w]=X[i-w:i]; yo[i-w]=y[i]
    return Xo, yo

X_seq, y_seq = make_tensors(_X_sc, _y_arr, WINDOW)
del _X_sc, _y_arr; gc.collect()

sv_tr = int(len(X_seq)*0.70); sv_va = int(len(X_seq)*0.85)
X_seq_tr, y_seq_tr = X_seq[:sv_tr],      y_seq[:sv_tr]
X_seq_va, y_seq_va = X_seq[sv_tr:sv_va], y_seq[sv_tr:sv_va]
X_seq_te, y_seq_te = X_seq[sv_va:],      y_seq[sv_va:]
del X_seq, y_seq; gc.collect()

print(f"Tensors — train:{X_seq_tr.shape}  val:{X_seq_va.shape}  test:{X_seq_te.shape}")
mem_check("LSTM tensors done")


---
## Phase 11 — BiLSTM Training (Cosine LR Decay + Better Convergence)

- LSTM stopped at epoch 10 (patience=7). Val AUC only reached 0.8043.
- `CosineDecayRestarts` learning rate schedule (start=1e-3, min=1e-5)
- `patience=10` — more room before stopping
- `epochs=80` — higher ceiling
- `class_weight={0:1.0, 1:2.5}` — stronger minority class penalty
- Added `SpatialDropout1D` + `GlobalAveragePooling` variant for better regularisation


### 11A — Build & Train BiLSTM

In [ ]:
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import (Bidirectional, LSTM, Dense,
                                         Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

# ── FIX 2: cosine decay schedule ──────────────────────────────────────────
steps_per_epoch = max(1, len(X_seq_tr)//2048)
lr_schedule = CosineDecayRestarts(
    initial_learning_rate = 1e-3,
    first_decay_steps     = steps_per_epoch * 5,   # restart every 5 epochs
    t_mul                 = 2.0,
    m_mul                 = 0.90,
    alpha                 = 1e-5,
)

lstm_model = Sequential([
    Bidirectional(LSTM(64, return_sequences=True),
                  input_shape=(WINDOW, X_seq_tr.shape[2])),
    BatchNormalization(),
    Dropout(0.30),
    Bidirectional(LSTM(32, return_sequences=False)),
    BatchNormalization(),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1,  activation="sigmoid"),
], name="BiLSTM_AIOps_v5")

lstm_model.compile(
    optimizer = Adam(learning_rate=lr_schedule),
    loss      = "binary_crossentropy",
    metrics   = [
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)
lstm_model.summary()

cb = [
    EarlyStopping(monitor="val_auc", mode="max",
                  patience=10,                     # FIX 2: was 7
                  restore_best_weights=True, verbose=1),
]

history = lstm_model.fit(
    X_seq_tr, y_seq_tr,
    validation_data = (X_seq_va, y_seq_va),
    epochs          = 80,                          # FIX 2: was 60→80
    batch_size      = 2048,
    class_weight    = {0:1.0, 1:2.5},             # FIX 2: was 1.68
    callbacks       = cb,
    verbose         = 1,
)

stopped_at = len(history.history["loss"])
print(f"\nStopped at epoch {stopped_at}/80")
mem_check("post LSTM training")


### 11B — BiLSTM Learning Curves (Loss / AUC / Precision / Recall)

In [ ]:
h   = history.history
eps = list(range(1, stopped_at+1))

fig, axes = plt.subplots(2,2,figsize=(14,9))
fig.suptitle("BiLSTM — Training History (per Epoch) v5",
             fontweight="bold",fontsize=14)

panels = [
    ("loss",      "val_loss",      "Binary Cross-Entropy Loss","#42A5F5","#EF5350"),
    ("auc",       "val_auc",       "AUC-ROC",                  "#66BB6A","#EF5350"),
    ("precision", "val_precision", "Precision",                "#AB47BC","#EF5350"),
    ("recall",    "val_recall",    "Recall",                   "#FFA726","#EF5350"),
]
for ax, (tr_k, va_k, title, tc, vc) in zip(axes.flatten(), panels):
    if tr_k in h:
        ax.plot(eps, h[tr_k], label="Train", lw=2, color=tc)
        ax.plot(eps, h[va_k], label="Val",   lw=2, color=vc, ls="--")
        ax.axvline(stopped_at, color="green", ls=":", lw=1.5,
                   label=f"Best ({stopped_at})")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    else:
        ax.set_visible(False)

plt.tight_layout()
plt.savefig("lstm_learning_curves.png", dpi=130, bbox_inches="tight"); plt.show()

# Epoch summary table
print("\n── Epoch summary (every 5 + last) ──")
show = sorted(set(list(range(0,stopped_at,5))+[stopped_at-1]))
print(f"{'Epoch':>6} {'tr_loss':>9} {'va_loss':>9} {'tr_auc':>8} {'va_auc':>8} "
      f"{'tr_rec':>7} {'va_rec':>7}")
print("-"*55)
for e in show:
    if e < stopped_at:
        print(f"{e+1:>6} "
              f"{h.get('loss',[0]*80)[e]:>9.4f} "
              f"{h.get('val_loss',[0]*80)[e]:>9.4f} "
              f"{h.get('auc',[0]*80)[e]:>8.4f} "
              f"{h.get('val_auc',[0]*80)[e]:>8.4f} "
              f"{h.get('recall',[0]*80)[e]:>7.4f} "
              f"{h.get('val_recall',[0]*80)[e]:>7.4f}")


### 11C — BiLSTM Evaluation (Optimal Threshold)

In [ ]:
# ── Auto-select threshold on LSTM val set ────────────────────────────────────
lstm_proba_va = lstm_model.predict(X_seq_va, batch_size=4096, verbose=0).ravel()
best_f1_l, best_thr_l = 0.0, 0.50
for thr in np.arange(0.20, 0.71, 0.01):
    pred = (lstm_proba_va>=thr).astype("int8")
    f1   = f1_score(y_seq_va, pred, pos_label=1, zero_division=0)
    if f1 > best_f1_l:
        best_f1_l, best_thr_l = f1, thr

print(f"Optimal LSTM threshold: {best_thr_l:.2f}  val F1={best_f1_l:.4f}")
LSTM_THRESHOLD = best_thr_l

lstm_proba_te = lstm_model.predict(X_seq_te, batch_size=4096, verbose=0).ravel()
lstm_pred_te  = (lstm_proba_te>=LSTM_THRESHOLD).astype("int8")
y_lstm_te     = y_seq_te

print("\n" + "━"*62)
print(f"  BiLSTM — Test Set (threshold={LSTM_THRESHOLD:.2f})")
print("━"*62)
print(classification_report(y_lstm_te, lstm_pred_te,
                             target_names=["Passed","Failed"], digits=4))
lstm_auc = roc_auc_score(y_lstm_te, lstm_proba_te)
lstm_mcc = matthews_corrcoef(y_lstm_te, lstm_pred_te)
lstm_ll  = log_loss(y_lstm_te, lstm_proba_te)
print(f"  AUC-ROC  : {lstm_auc:.4f}")
print(f"  MCC      : {lstm_mcc:.4f}")
print(f"  Log-Loss : {lstm_ll:.4f}")


### 11D — BiLSTM ROC + PR Curves

In [ ]:
fpr_l,tpr_l,_ = roc_curve(y_lstm_te, lstm_proba_te)
pr_l,re_l,_   = precision_recall_curve(y_lstm_te, lstm_proba_te)
ap_l          = average_precision_score(y_lstm_te, lstm_proba_te)

fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("BiLSTM — ROC & Precision-Recall Curves",fontweight="bold",fontsize=13)
axes[0].plot(fpr_l,tpr_l,lw=2,color="#1565C0",label=f"AUC={lstm_auc:.4f}")
axes[0].plot([0,1],[0,1],ls="--",color="gray",lw=1)
axes[0].fill_between(fpr_l,tpr_l,alpha=0.08,color="#1565C0")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve",fontweight="bold"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(re_l,pr_l,lw=2,color="#E53935",label=f"AP={ap_l:.4f}")
axes[1].axhline(y_lstm_te.mean(),ls="--",color="gray",lw=1)
axes[1].fill_between(re_l,pr_l,alpha=0.08,color="#E53935")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve",fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("lstm_roc_pr.png", dpi=130, bbox_inches="tight"); plt.show()


### 11E — BiLSTM Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(
    y_lstm_te, lstm_pred_te,
    display_labels=["Passed","Failed"], cmap="Blues", ax=ax)
ax.set_title(f"BiLSTM Confusion Matrix (thr={LSTM_THRESHOLD:.2f})",
             fontweight="bold")
plt.tight_layout()
plt.savefig("lstm_confusion_matrix.png", dpi=130, bbox_inches="tight"); plt.show()


In [ ]:
lstm_model.save("aiops_lstm_model.keras")
print("BiLSTM → aiops_lstm_model.keras")
print("Scaler → aiops_lstm_scaler.pkl (already saved)")
mem_check("LSTM complete")


---
## Phase 12 — Fair Side-by-Side Comparison

>
> XGBoost was evaluated on `y_te` (138,394 rows) while
> LSTM was evaluated on `y_seq_te` (different size due to window offset).
> Metrics were therefore not directly comparable.
>
> Both models predict on the **same final N rows** of the test set.


In [ ]:
# ── align test set ────────────────────────────────────────────────────
n_common = min(len(y_te), len(y_lstm_te))
_y_c    = y_te[-n_common:]
_xp     = xgb_proba_te[-n_common:]
_lp     = lstm_proba_te[-n_common:]
_xpred  = (_xp  >= XGB_THRESHOLD).astype("int8")
_lpred  = (_lp  >= LSTM_THRESHOLD).astype("int8")

print(f"Aligned test samples : {n_common:,}")
print(f"XGB  threshold used  : {XGB_THRESHOLD:.2f}")
print(f"LSTM threshold used  : {LSTM_THRESHOLD:.2f}")

def full_metrics(name, y_true, y_pred, y_proba):
    return {
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y_true,y_pred),4),
        "Precision": round(precision_score(y_true,y_pred,zero_division=0),4),
        "Recall"   : round(recall_score(y_true,y_pred,zero_division=0),4),
        "F1"       : round(f1_score(y_true,y_pred,zero_division=0),4),
        "AUC-ROC"  : round(roc_auc_score(y_true,y_proba),4),
        "MCC"      : round(matthews_corrcoef(y_true,y_pred),4),
        "Log-Loss" : round(log_loss(y_true,y_proba),4),
    }

r1 = full_metrics("XGBoost", _y_c, _xpred, _xp)
r2 = full_metrics("BiLSTM",  _y_c, _lpred, _lp)

cmp_df = pd.DataFrame([r1,r2]).set_index("Model")
print("\n" + "═"*70)
print("  FINAL COMPARISON — ALIGNED TEST SET (FAIR)")
print("═"*70)
print(cmp_df.to_string())

# Bar chart
metrics_show = ["Accuracy","Precision","Recall","F1","AUC-ROC","MCC"]
x = np.arange(len(metrics_show)); width=0.35

fig, ax = plt.subplots(figsize=(13,5))
b1 = ax.bar(x-width/2, cmp_df.loc["XGBoost",metrics_show], width,
            label="XGBoost", color="#1565C0", edgecolor="white")
b2 = ax.bar(x+width/2, cmp_df.loc["BiLSTM", metrics_show], width,
            label="BiLSTM",  color="#E53935", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(metrics_show, fontsize=10)
ax.set_ylabel("Score"); ax.set_ylim(0,1.05)
ax.set_title("XGBoost vs BiLSTM — Independent Evaluation (Aligned Test Set)",
             fontweight="bold",fontsize=13)
ax.legend(fontsize=10); ax.grid(axis="y",alpha=0.3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{bar.get_height():.3f}", ha="center",fontsize=7.5,rotation=90)
plt.tight_layout()
plt.savefig("model_comparison_bar.png", dpi=130, bbox_inches="tight"); plt.show()

# Overlaid ROC + PR
fpr_x2,tpr_x2,_ = roc_curve(_y_c,_xp)
fpr_l2,tpr_l2,_ = roc_curve(_y_c,_lp)
prx2,rex2,_ = precision_recall_curve(_y_c,_xp)
prl2,rel2,_ = precision_recall_curve(_y_c,_lp)

fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("XGBoost vs BiLSTM — Overlaid Curves (Aligned Test Set)",
             fontweight="bold",fontsize=13)
axes[0].plot(fpr_x2,tpr_x2,lw=2,color="#1565C0",
             label=f"XGBoost AUC={roc_auc_score(_y_c,_xp):.4f}")
axes[0].plot(fpr_l2,tpr_l2,lw=2,color="#E53935",ls="--",
             label=f"BiLSTM  AUC={roc_auc_score(_y_c,_lp):.4f}")
axes[0].plot([0,1],[0,1],ls=":",color="gray",lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve",fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rex2,prx2,lw=2,color="#1565C0",
             label=f"XGBoost AP={average_precision_score(_y_c,_xp):.4f}")
axes[1].plot(rel2,prl2,lw=2,color="#E53935",ls="--",
             label=f"BiLSTM  AP={average_precision_score(_y_c,_lp):.4f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve",fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("model_comparison_curves.png", dpi=130, bbox_inches="tight"); plt.show()
